# ERP003950: набор истины после фильтрации → PCR1 → фрагментация → PCR2 → NovaSeq PE150

Ветка предобработки: `results/ERP003950/`.

Биологический набор истины берётся только из её собственного `post_annotation_filtered/airr_pass`.

В ERP003950 нет UMI, поэтому точные V…J-последовательности дедуплицируются внутри каждого sample, а начальная abundance по умолчанию равна `one_per_unique`. Наблюдаемая multiplicity сохраняется только для QC и анализа чувствительности.

**Фрагментация:** `uniform single random-cut`.


## Структура выходных данных

`results/ERP003950/simulated/`
`insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut/`

Внутри:
- `00_primary_truth/`
- `01_pcr1/`
- `02_fragmentation/`
- `03_pcr2/`
- `04_read_allocation/`
- `05_fastq_native/`
- `06_fastq_pe150/`
- `logs/`
- `qc/`

Для FastQC/MultiQC общий `qc.ipynb` использует:
`DATASET="ERP003950"`, `STAGE="simulated"` и имя этой ветки.


## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next(
    (p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))),
    None,
)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"

for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

for tool in ("iss",):
    path = shutil.which(tool)
    if not path:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {path}")

print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut"

# ---------------------------------------------------------------------
# Эталон, входная порция библиотеки и глубина (входят в имя ветки)
# ---------------------------------------------------------------------
# amplicon_with_primers: физический ампликон PCR1 = склеенный рид (V-праймер
#   в нём сохранён, остаток до V, V…J, C-фрагмент) + revcomp(C-праймер IgG).
#   C-праймер вырезан cutadapt на стадии pr_trimmed (primer_trim_mouse_fastp_q30_u40).
# vj_only: эталон обрезан ровно до V…J.
TRUTH_TEMPLATE_MODE = "amplicon_with_primers"
# Тот же олигонуклеотид, что FORWARD_ADAPTER в primer_trim_mouse.ipynb
# (читается от C-области внутрь ампликона, поэтому в верхней цепи стоит его revcomp).
MOUSE_C_PRIMER = "CARKGGATRRRCHGATGGGG"
# Масса PCR1-продукта, идущая на фрагментацию (протокол 5′MTPX: 5–10 нг). Число
# молекул порции = масса / (средняя длина ампликона · 650 г/моль) · N_A, то есть
# тысячи копий каждой исходной молекулы, как в реальной подготовке библиотеки.
LIBRARY_INPUT_NG = 5.0
# UMI нет, поэтому ошибки секвенирования исходных ридов убираются схлопыванием: точный
# вариант V…J поглощается соседом, отличающимся одной заменой и встречающимся
# >= ERROR_COLLAPSE_MIN_RATIO раз чаще; риды поглощённого добавляются к копиям соседа.
ERROR_COLLAPSE_MAX_MISMATCHES = 1
ERROR_COLLAPSE_MIN_RATIO = 3
# Шаблон проверен (входит в оценку сборщиков), если после схлопывания у него >= 2 ридов;
# непроверенные шаблоны симулируются как физические молекулы, но не оцениваются.
VERIFIED_MIN_READS = 2
# Множитель глубины относительно исходных сырых пар: исходный MiSeq читал
# неразрезанные ампликоны, после фрагментации на молекулу нужно несколько пар.
READ_BUDGET_MULTIPLIER = 3.0

# Мягкий size-selection после фрагментации: центр распределения длин вставок (нт).
# Входит в имя ветки. 250 нт для ~430–500-нт ампликона даёт перекрывающиеся 5′/3′-куски
# и заметную долю пар с несеквенированным промежутком между R1 и R2 (mate-linked).
SIZE_SELECTION_TARGET = 250

_TRUTH_TAG = {"amplicon_with_primers": "amp", "vj_only": "vj"}[TRUTH_TEMPLATE_MODE]
_TRUTH_TAG += f"_ec{ERROR_COLLAPSE_MAX_MISMATCHES}x{ERROR_COLLAPSE_MIN_RATIO:g}"
BRANCH_NAME = f"{BRANCH_NAME}_{_TRUTH_TAG}_in{LIBRARY_INPUT_NG:g}ng_rb{READ_BUDGET_MULTIPLIER:g}x_ss{SIZE_SELECTION_TARGET}"
SOURCE_STAGE = "post_annotation_filtered"

RESULT_DATASET = "ERP003950"
RAW_DATASET = "ERP003950"

SAMPLES = [
    "ERR346596",
    "ERR346597",
    "ERR346598",
    "ERR346599",
    "ERR346600",
    "ERR346601",
]

RUN_LOCUS = {sample: "IGH" for sample in SAMPLES}

def resolve_volume():
    candidates = []

    env_root = os.environ.get("BCR_VOLUME")
    if env_root:
        candidates.append(Path(env_root))

    candidates.extend([
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ])

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    seen = set()

    for root in candidates:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)

        if (
            (root / "results" / RESULT_DATASET).is_dir()
            and (root / "raw" / RAW_DATASET).is_dir()
        ):
            return root

    raise FileNotFoundError(
        f"Cannot locate results/{RESULT_DATASET} together with raw/{RAW_DATASET}. "
        "Set BCR_VOLUME explicitly."
    )

VOLUME = resolve_volume()
DATASET_DIR = VOLUME / "results" / RESULT_DATASET

POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / RAW_DATASET

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run the fastp_q30_u40 mouse post-annotation filter first."
    )

FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == RESULT_DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS

MIN_TEMPLATE_LENGTH = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"

TEMPLATES_DIR = TRUTH_DIR
COUNTS_DIR = ALLOCATION_DIR

for directory in (
    TRUTH_DIR,
    PCR1_DIR,
    FRAGMENTATION_DIR,
    PCR2_DIR,
    ALLOCATION_DIR,
    FASTQ_NATIVE_DIR,
    FASTQ_DIR,
    MODEL_DIR,
    LOGS_DIR,
    QC_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

NPROC = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False
COMPRESS = True

DROP_SEQUENCES_WITH_N = True
# Исходные копии шаблона = его риды после схлопывания ошибок (UMI нет).
STARTING_COPIES_MODE = "observed_multiplicity"

PCR_CYCLES = 25
PCR_EFFICIENCY_MEAN = 0.85
PCR_EFFICIENCY_CONCENTRATION = 80.0
PCR_MAX_COPIES = 10**12

# Фрагментация и отбор библиотеки.
#
# Из PCR1-пула берётся конечная порция молекул, каждая молекула физически
# разрезается; до size-selection проверяется сохранение нуклеотидной массы.
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "uniform_single_cut"


# После разрезания выполняется отдельный мягкий size-selection
# (SIZE_SELECTION_TARGET ± SIZE_SELECTION_SD).
# SIZE_SELECTION_TARGET задан в начале ячейки и входит в имя ветки.
SIZE_SELECTION_SD = 40
# ISS 2.0.1 simulate_read: assert read_length < len(template); NovaSeq profile read_length = 151 nt.
MIN_SEQUENCABLE_FRAGMENT_LENGTH = 152
# Перевод массы порции в молекулы: dsDNA ~650 г/моль на пару оснований.
DSDNA_G_PER_MOL_PER_BP = 650.0
AVOGADRO = 6.02214076e23
# Куски (start, end) с ожидаемым числом < порога на молекулу не учитываются;
# их суммарная доля пишется в fragment_length_model.tsv.
FRAGMENT_MODEL_MIN_EXPECTED = 1e-12

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "library_input": "LIBRARY_INPUT_NG of PCR1 product: molecules = ng / (PCR1-weighted mean length * 650 g/mol) * N_A, multinomial by PCR1 abundance",
    "library_input_ng": LIBRARY_INPUT_NG,
    "fragment_pool": "per template with reads: Poisson(aliquot copies * expected size-selected (start, end) fragments per molecule of its length), PCR2 per fragment species, reads multinomial by PCR2 copies; reads are split across templates by expected size-selected fragments",
    "breakpoints": "exactly one uniformly sampled inter-base bond per library-input molecule",
    "daughter_fragments": "both physical daughters are created before size selection",
    "size_selection": {
        "type": "Gaussian acceptance kernel",
        "target_nt": SIZE_SELECTION_TARGET,
        "sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    },
    "scientific_analogs": [
        "BEERS2: default sequence-independent uniform bond breaking + separate size selection; DOI 10.1093/bib/bbae164",
        "Flux Simulator: modular fragmentation + size selection; DOI 10.1093/nar/gks666",
    ],
}

LIBRARY_PCR_CYCLES = 10
LIBRARY_PCR_EFFICIENCY_MEAN = 0.85
LIBRARY_PCR_EFFICIENCY_CONCENTRATION = 80.0
LIBRARY_PCR_MAX_COPIES = 10**15

READ_BUDGET_MODE = "match_raw_pairs"
FIXED_READ_PAIRS = 500_000

SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "InSilicoSeq bundled NovaSeq profile",
    "conditioned_on_post_annotation_filter": False,
}

print("VOLUME:", VOLUME)
print("RESULT_DATASET:", RESULT_DATASET)
print("RAW_DATASET:", RAW_DATASET)
print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# ---------------------------------------------------------------------
# Очистка существующих сгенерированных артефактов
# ---------------------------------------------------------------------
# Удаляться могут только файлы внутри этой ветки симуляции.
# Исходные raw, filtered и annotation данные не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [
        TRUTH_DIR,
        PCR1_DIR,
        FRAGMENTATION_DIR,
        PCR2_DIR,
        ALLOCATION_DIR,
        FASTQ_NATIVE_DIR,
        FASTQ_DIR,
        MODEL_DIR,
        QC_DIR,
        LOGS_DIR,
    ]

    for directory in generated_dirs:
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        directory.mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение набора истины: ампликон PCR1

`post_annotation_filtered/airr_pass` является источником данных (читается `.airr.tsv` или `.airr.tsv.gz`).

Для каждого sample ноутбук:
1. повторно проверяет условия фильтра с помощью assertions;
2. схлопывает точные наблюдаемые V…J-последовательности (`v_sequence_start:j_sequence_end`) внутри sample; SHM сохраняются;
3. исключает записи с неоднозначными N (в режиме ампликона — во всём склеенном риде);
4. при `TRUTH_TEMPLATE_MODE = "amplicon_with_primers"` строит физический ампликон PCR1: самые частые для этого V…J фланги склеенного рида (V-праймер уже сохранён в риде, C-фрагмент) + revcomp C-праймера IgG, вырезанного cutadapt. Координаты V…J записаны в `vj_start`/`vj_end` (0-based, half-open) — этот интервал является эталоном для оценки сборщиков;
5. схлопывает ошибки секвенирования исходных ридов: вариант поглощается соседом той же длины, отличающимся одной заменой и встречающимся ≥ `ERROR_COLLAPSE_MIN_RATIO` раз чаще; его риды добавляются к соседу;
6. число исходных копий шаблона = его риды после схлопывания (UMI нет);
7. шаблон **проверен** (`verified`), если после схлопывания у него ≥ `VERIFIED_MIN_READS` ридов; непроверенные шаблоны симулируются, но не входят в оценку сборщиков. CDR3 (`junction`) записывается в `template_qc.tsv`.


In [ ]:
def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

_IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT",
          "K": "GT", "M": "AC", "B": "CGT", "D": "AGT", "H": "ACT", "V": "ACG", "N": "ACGT"}
_RC = str.maketrans("ACGT", "TGCA")

def _revcomp(s):
    return s.translate(_RC)[::-1]

def _truth_airr_path(sample):
    for name in (f"{sample}.airr.tsv", f"{sample}.airr.tsv.gz"):
        if (TRUTH_AIRR_DIR / name).exists():
            return TRUTH_AIRR_DIR / name
    raise FileNotFoundError(f"Filtered AIRR missing: {TRUTH_AIRR_DIR}/{sample}.airr.tsv[.gz]")

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(row, sample):
    problems = []

    v_start = _as_int(row.get("v_sequence_start"))
    j_end = _as_int(row.get("j_sequence_end"))
    v_germline_start = _as_int(row.get("v_germline_start"))
    v_identity = _as_float(row.get("v_identity"))
    j_identity = _as_float(row.get("j_identity"))
    v_support = _as_float(row.get("v_support"))
    j_support = _as_float(row.get("j_support"))

    if (
        v_start is None
        or j_end is None
        or j_end - v_start + 1 < MIN_TEMPLATE_LENGTH
    ):
        problems.append("V-J span")

    if v_germline_start != 1:
        problems.append("V 5-prime completeness")

    if not _truthy(row.get("complete_vdj")):
        problems.append("complete_vdj")

    if str(row.get("locus", "")).strip() != RUN_LOCUS[sample]:
        problems.append("locus")

    if not str(row.get("v_call", "")).strip():
        problems.append("v_call")
    if not str(row.get("j_call", "")).strip():
        problems.append("j_call")

    if not _truthy(row.get("productive")):
        problems.append("productive")

    if not _falsey(row.get("stop_codon")):
        problems.append("stop_codon")

    if v_identity is None or v_identity < MIN_V_IDENTITY:
        problems.append("v_identity")
    if j_identity is None or j_identity < MIN_J_IDENTITY:
        problems.append("j_identity")
    if v_support is None or v_support > MAX_V_SUPPORT:
        problems.append("v_support")
    if j_support is None or j_support > MAX_J_SUPPORT:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{sample} airr_pass violates filter_summary contract for "
            f"{row.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    return {
        "dataset": RESULT_DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "truth_template_mode": TRUTH_TEMPLATE_MODE,
            "c_primer": MOUSE_C_PRIMER if TRUTH_TEMPLATE_MODE == "amplicon_with_primers" else None,
            "flanks": "most frequent (leader incl. retained V primer, C stub) among member reads + revcomp(C primer)",
            "vj_coordinates": "template_qc.tsv vj_start/vj_end, 0-based half-open",
            "collapse_scope": "within_sample",
            "collapse_key": ["VJ_sequence"],
            "starting_copies_mode": STARTING_COPIES_MODE,
            "dataset_has_umi": False,
            "error_collapse": {"max_mismatches": ERROR_COLLAPSE_MAX_MISMATCHES,
                               "min_parent_ratio": ERROR_COLLAPSE_MIN_RATIO,
                               "scope": "within_sample, same length, one level, directional"},
            "verified": f"reads after collapse >= {VERIFIED_MIN_READS}",
            "junction": "most frequent AIRR junction span among member reads",
        },
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
        "branch": BRANCH_NAME,
        "library_input_ng": LIBRARY_INPUT_NG,
        "read_budget_multiplier": READ_BUDGET_MULTIPLIER,
        "seed": SEED,
    }

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False

    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and "
            "CLEAN_OLD_OUTPUTS=True, or choose a new branch."
        )
    return True

def _one_mismatch(a, b):
    d = 0
    for x, y in zip(a, b):
        if x != y:
            d += 1
            if d > 1:
                return False
    return True

def collapse_errors(counts):
    """child -> parent: вариант V…J поглощается соседом той же длины, отличающимся ровно
    одной заменой и встречающимся >= ERROR_COLLAPSE_MIN_RATIO раз чаще (самым частым из них).

    Кандидаты ищутся по половинам последовательности: при одной замене одна половина
    совпадает точно. Варианты обходятся от частых к редким; поглощённый вариант сам
    родителем не становится (одноуровневое, направленное схлопывание).
    """
    assert ERROR_COLLAPSE_MAX_MISMATCHES == 1
    index, parent = defaultdict(list), {}
    for s in sorted(counts, key=lambda x: (-counts[x], x)):
        L, h, n = len(s), len(s) // 2, counts[s]
        best = None
        for key, lo, hi in (((L, 0, s[:h]), h, L), ((L, 1, s[h:]), 0, h)):
            for p in index.get(key, ()):
                if (counts[p] >= ERROR_COLLAPSE_MIN_RATIO * n and _one_mismatch(s[lo:hi], p[lo:hi])
                        and (best is None or counts[p] > counts[best])):
                    best = p
        if best is not None:
            parent[s] = best
        elif n >= ERROR_COLLAPSE_MIN_RATIO:
            index[(L, 0, s[:h])].append(s)
            index[(L, 1, s[h:])].append(s)
    return parent

# самопроверка: 3 рида при соседе с 9 поглощаются; сосед с 4 < 3·9 не поглощает 9;
# вариант в 2 заменах от родителя остаётся; поглощённый вариант не становится родителем.
assert collapse_errors({"AAAAAAAA": 9, "AAAAAAAT": 3, "AAAAAATT": 1, "CAAAAAAA": 4}) == {"AAAAAAAT": "AAAAAAAA"}

def build_sample_templates(sample, force=FORCE):
    out_fasta = TRUTH_DIR / f"{sample}_templates.fasta"
    out_tsv = TRUTH_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / f"{sample}_source_truth_audit.tsv"

    if (
        out_fasta.exists()
        and out_tsv.exists()
        and audit_path.exists()
        and not force
    ):
        _check_cached_truth_provenance()
        print(f"[{sample}] [skip] post-filter truth exists")
        return

    airr = _truth_airr_path(sample)
    filtered_fastq = FILTERED_FASTQ_DIR / f"{sample}_filtered.fastq.gz"

    if not filtered_fastq.exists():
        raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")

    expected_passed = int(FILTER_SUMMARY["samples"][sample]["passed"])

    counts = defaultdict(int)
    observed = defaultdict(int)
    flanks = defaultdict(lambda: defaultdict(int))
    junctions = defaultdict(Counter)

    with open_text(airr) as handle:
        reader = csv.DictReader(handle, delimiter="\t")

        for row in reader:
            counts["airr_pass_rows"] += 1
            _validate_post_filter_row(row, sample)

            start = int(row["v_sequence_start"]) - 1
            end = int(row["j_sequence_end"])
            full = (row.get("sequence") or "").upper()
            seq = full[start:end]

            if len(seq) < MIN_TEMPLATE_LENGTH:
                raise RuntimeError(
                    f"{sample}: cropped V-J sequence shorter than filter threshold "
                    f"for {row.get('sequence_id')}"
                )

            if DROP_SEQUENCES_WITH_N and "N" in (full if TRUTH_TEMPLATE_MODE == "amplicon_with_primers" else seq):
                counts["simulation_excluded_N"] += 1
                continue

            observed[seq] += 1
            flanks[seq][(full[:start], full[end:])] += 1
            # junction = sequence[cdr3_start-4 : cdr3_end+3] (AIRR, 1-based), в координатах V…J
            if row.get("junction") and row.get("cdr3_start"):
                junctions[seq][(int(row["cdr3_start"]) - 4 - start, len(row["junction"]))] += 1
            counts["simulation_eligible_rows"] += 1

    if counts["airr_pass_rows"] != expected_passed:
        raise RuntimeError(
            f"{sample}: airr_pass rows={counts['airr_pass_rows']:,}, "
            f"but filter_summary passed={expected_passed:,}"
        )

    # Схлопывание ошибок секвенирования: риды поглощённого варианта уходят соседу.
    parent_of = collapse_errors(observed)
    copies = dict(observed)
    absorbed = defaultdict(int)
    for child, parent in parent_of.items():
        copies[parent] += observed[child]
        absorbed[parent] += 1
        del copies[child]

    ordered = sorted(
        copies.items(),
        key=lambda item: (-item[1], item[0]),
    )

    iupac_rng = np.random.default_rng(sample_seed(sample, 500))
    amplicon = TRUTH_TEMPLATE_MODE == "amplicon_with_primers"

    def _junction(seq, spans):
        if not spans:
            return ""
        jo, jl = spans.most_common(1)[0][0]
        return seq[jo:jo + jl] if 0 <= jo and jo + jl <= len(seq) else ""

    with open(out_fasta, "w") as fasta, open(out_tsv, "w", newline="") as table:
        fields = [
            "template_id",
            "locus",
            "length",
            "vj_start",
            "vj_end",
            "observed_multiplicity",
            "exact_reads",
            "absorbed_variants",
            "verified",
            "junction",
        ]
        writer = csv.DictWriter(table, fieldnames=fields, delimiter="\t")
        writer.writeheader()

        for i, (seq, multiplicity) in enumerate(ordered, 1):
            template_id = f"{sample}_IGH_tpl_{i:07d}"
            if amplicon:
                # самые частые фланги среди ридов этого V…J; вырожденные позиции
                # праймера разрешаются воспроизводимо
                lead, cstub = max(flanks[seq].items(), key=lambda kv: kv[1])[0]
                c_primer = "".join(
                    c if c in "ACGT" else str(iupac_rng.choice(list(_IUPAC[c])))
                    for c in MOUSE_C_PRIMER
                )
                template = lead + seq + cstub + _revcomp(c_primer)
            else:
                lead, template = "", seq
            fasta.write(f">{template_id}\n{template}\n")
            writer.writerow({
                "template_id": template_id,
                "locus": "IGH",
                "length": len(template),
                "vj_start": len(lead),
                "vj_end": len(lead) + len(seq),
                "observed_multiplicity": multiplicity,
                "exact_reads": observed[seq],
                "absorbed_variants": absorbed[seq],
                "verified": multiplicity >= VERIFIED_MIN_READS,
                "junction": _junction(seq, junctions[seq]),
            })
    audit = {
        "sample": sample,
        "expected_passed_from_filter_summary": expected_passed,
        "airr_pass_rows": counts["airr_pass_rows"],
        "simulation_eligible_rows": counts["simulation_eligible_rows"],
        "simulation_excluded_N": counts["simulation_excluded_N"],
        "unique_exact_vj": len(observed),
        "absorbed_error_variants": len(parent_of),
        "templates_after_collapse": len(ordered),
        "verified_templates": sum(n >= VERIFIED_MIN_READS for n in copies.values()),
    }
    with open(audit_path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(audit), delimiter="\t")
        writer.writeheader()
        writer.writerow(audit)

    print(
        f"[{sample}] airr_pass={counts['airr_pass_rows']:,}; "
        f"simulation_eligible={counts['simulation_eligible_rows']:,}; "
        f"unique_VJ={len(observed):,}; after_collapse={len(ordered):,}; "
        f"excluded_N={counts['simulation_excluded_N']:,}"
    )

for sample in SAMPLES:
    build_sample_templates(sample)

PROVENANCE_PATH.write_text(
    json.dumps(_current_provenance(), indent=2) + "\n"
)
print("provenance:", PROVENANCE_PATH)


## 5. Сводка QC точного набора истины V…J


In [ ]:
template_summary = []

for sample in SAMPLES:
    path = TRUTH_DIR / f"{sample}_template_qc.tsv"
    rows = list(csv.DictReader(open(path), delimiter="\t"))
    multiplicities = np.asarray(
        [int(row["observed_multiplicity"]) for row in rows],
        dtype=np.int64,
    )

    template_summary.append({
        "sample": sample,
        "locus": "IGH",
        "unique_templates": len(rows),
        "filtered_reads_represented": int(multiplicities.sum()),
        "singleton_templates": int((multiplicities == 1).sum()),
        "verified_templates": sum(row["verified"] == "True" for row in rows),
        "max_observed_multiplicity": (
            int(multiplicities.max()) if len(multiplicities) else 0
        ),
    })

out = QC_DIR / "template_summary.tsv"
with open(out, "w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(template_summary[0]),
        delimiter="\t",
    )
    writer.writeheader()
    writer.writerows(template_summary)

print(*template_summary, sep="\n")
print("wrote", out)


## 6. Встроенная модель ошибок NovaSeq из InSilicoSeq

Эта ветка не обучает эмпирическую модель ошибок на ERP003950. Используется общий
встроенный профиль NovaSeq, что делает слой ошибок секвенирования сопоставимым между датасетами.


In [ ]:
import iss
ISS_PROFILE_NAME="NovaSeq"; ISS_NATIVE_READ_LENGTH=151
profile_path=Path(iss.__file__).resolve().parent/"profiles"/ISS_PROFILE_NAME
with np.load(profile_path,allow_pickle=True) as z:
    assert int(z["read_length"])==ISS_NATIVE_READ_LENGTH
    print({"profile":ISS_PROFILE_NAME,"model":str(z["model"].item()),"native_read_length":int(z["read_length"]),"target":TARGET_READ_LENGTH})


In [ ]:
# В этой ветке модель не обучается: используется встроенный KDE-профиль NovaSeq.

In [ ]:
# Происхождение модели записывается в ячейке параметров выше.

## 7. Бюджет секвенирования для каждого sample


In [ ]:
def get_read_budget(sample):
    if READ_BUDGET_MODE == "fixed":
        return int(FIXED_READ_PAIRS)

    if READ_BUDGET_MODE == "match_raw_pairs":
        r1 = RAW_FASTQ_DIR / f"{sample}_1.fastq.gz"
        r2 = RAW_FASTQ_DIR / f"{sample}_2.fastq.gz"

        if not r1.exists() or not r2.exists():
            raise FileNotFoundError(
                f"Missing raw pair for {sample}: {r1}, {r2}"
            )

        n1 = count_fastq(r1)
        n2 = count_fastq(r2)

        if n1 != n2:
            raise ValueError(
                f"{sample}: R1/R2 count mismatch: {n1:,} vs {n2:,}"
            )

        return int(round(n1 * READ_BUDGET_MULTIPLIER))

    raise ValueError(f"Unknown READ_BUDGET_MODE={READ_BUDGET_MODE!r}")

READ_BUDGETS = {sample: get_read_budget(sample) for sample in SAMPLES}
print(READ_BUDGETS)


## 8. PCR1 для точных V…J templates после фильтрации


In [ ]:
def load_dedup_templates(sample):
    fasta = TRUTH_DIR / f"{sample}_templates.fasta"
    qc = TRUTH_DIR / f"{sample}_template_qc.tsv"

    meta = {
        row["template_id"]: row
        for row in csv.DictReader(open(qc), delimiter="\t")
    }

    rows = []
    for template_id, seq in iter_fasta(fasta):
        m = meta[template_id]
        rows.append({
            "template_id": template_id,
            "sequence": seq,
            "length": len(seq),
            "vj_start": int(m.get("vj_start") or 0),
            "vj_end": int(m.get("vj_end") or len(seq)),
            "locus": m["locus"],
            "observed_multiplicity": int(m["observed_multiplicity"]),
            "verified": m["verified"] == "True",
        })

    return rows

def starting_copies(row):
    if STARTING_COPIES_MODE == "one_per_unique":
        return 1

    if STARTING_COPIES_MODE == "observed_multiplicity":
        return row["observed_multiplicity"]

    raise ValueError(STARTING_COPIES_MODE)

def branching_pcr(n0, efficiency, cycles, rng):
    n = int(n0)

    for _ in range(cycles):
        if n <= 0:
            return 0

        n += int(rng.binomial(n, efficiency))

        if n >= PCR_MAX_COPIES:
            return int(PCR_MAX_COPIES)

    return n

def simulate_pcr_pool(sample):
    rng = np.random.default_rng(sample_seed(sample, 1000))
    rows = load_dedup_templates(sample)

    mean = PCR_EFFICIENCY_MEAN
    concentration = PCR_EFFICIENCY_CONCENTRATION

    if not (0 < mean <= 1):
        raise ValueError("PCR_EFFICIENCY_MEAN must be in (0,1]")

    if mean == 1:
        alpha = beta = None
    else:
        alpha = mean * concentration
        beta = (1 - mean) * concentration

    for row in rows:
        efficiency = (
            1.0
            if mean == 1
            else float(rng.beta(alpha, beta))
        )

        n0 = starting_copies(row)
        n_pcr = branching_pcr(
            n0,
            efficiency,
            PCR_CYCLES,
            rng,
        )

        row.update(
            starting_copies=n0,
            pcr_efficiency=efficiency,
            pcr_copies=n_pcr,
        )

    return rows

_test_rng = np.random.default_rng(1)
assert branching_pcr(1, 1.0, 10, _test_rng) == 2**10
assert branching_pcr(1, 0.0, 10, _test_rng) == 1
print("branching PCR smoke tests: OK")


## 9. Фрагментация: baseline random-cut

1. Из PCR1-пула берётся порция `LIBRARY_INPUT_NG` нг пропорционально численности шаблонов после PCR1.
2. Для каждой выбранной физической молекулы выбирается **ровно одна** межнуклеотидная связь с равной вероятностью среди всех `L-1` возможных позиций.
3. Молекула действительно делится на **два дочерних фрагмента**.
4. После физического разрыва выполняется отдельный мягкий size-selection вокруг `SIZE_SELECTION_TARGET` nt (по умолчанию 250) (`SD=40`).
5. В PCR2 переходят только фрагменты, прошедшие size-selection и достаточно длинные для используемого профиля секвенирования.


Научная мотивация: BEERS2 использует sequence-independent uniform bond breaking как базовый режим fragmentation и отделяет fragmentation от size selection (DOI `10.1093/bib/bbae164`). Flux Simulator также моделирует fragmentation и size selection как отдельные стадии (DOI `10.1093/nar/gks666`).

**Реализация (порция `LIBRARY_INPUT_NG`).** В фрагментацию идут 5 нг продукта PCR1 — тысячи копий каждой исходной молекулы. Перечислять миллиарды кусков не нужно: фрагментация не зависит от последовательности, поэтому для каждой длины шаблона один раз считается ожидаемое число кусков `(start, end)` на молекулу (с сохранением массы нуклеотидов). Затем только для шаблонов, получивших риды, пул кусков разыгрывается как Poisson(копии в порции × ожидаемое число), проходит PCR2 (эффективность своя у каждого вида куска), и риды шаблона распределяются по копиям после PCR2. Таблица `02_fragmentation/<sample>_fragment_length_model.tsv` — модель по длинам, `<sample>_library_summary.tsv` — сводка порции.


In [ ]:
def _size_selection_probability(length):
    """Probability that a physical fragment enters the PCR2 library (scalar or array)."""
    length = np.asarray(length, dtype=float)
    z = (length - SIZE_SELECTION_TARGET) / SIZE_SELECTION_SD
    return np.where(length < MIN_SEQUENCABLE_FRAGMENT_LENGTH, 0.0, np.exp(-0.5 * z * z))


FRAGMENT_ID_TAG = "rc"


def expected_terminal_fragments(L):
    """Ожидаемое число концевых кусков (start, end) на одну молекулу длины L.

    Один равновероятный разрез по одной из L-1 связей: разрез c даёт [0, c) и [c, L)
    с вероятностью 1/(L-1) каждый.
    """
    c = np.arange(1, L)
    w = np.full(L - 1, 1.0 / (L - 1))
    return (np.concatenate([np.zeros_like(c), c]),
            np.concatenate([c, np.full_like(c, L)]),
            np.concatenate([w, w]))


# самопроверка: сохраняется масса нуклеотидов, на молекулу ровно 2 куска
_s, _e, _E = expected_terminal_fragments(450)
assert abs(float((_E * (_e - _s)).sum()) - 450) < 1e-9 and abs(float(_E.sum()) - 2) < 1e-9
print("fragmentation model self-check: OK")


## 10. PCR2 для фрагментов

PCR2 выполняется внутри этапа библиотеки (раздел 11): стохастическое ветвление для каждого вида куска со своей эффективностью.


In [ ]:
def branching_pcr_vectorized(n0, efficiency, cycles, max_copies, rng):
    n = np.asarray(n0, dtype=np.int64).copy()
    eff = np.asarray(efficiency, dtype=np.float64)
    for _ in range(cycles):
        active = n > 0
        if not active.any():
            break
        new = np.zeros_like(n)
        new[active] = rng.binomial(n[active], eff[active])
        n = n + new
        np.minimum(n, max_copies, out=n)
    return n


_test_rng = np.random.default_rng(1)
_vec = branching_pcr_vectorized(
    np.array([1]), np.array([1.0]), 10, 10**15, _test_rng
)
assert int(_vec[0]) == 2**10
print("vectorized branching PCR smoke test: OK")


## 11. PCR1 → порция → фрагментация → PCR2 → распределение ридов


In [ ]:
def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(
            {field: row[field] for field in fields}
            for row in rows
        )
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"

    if out.exists() and not force:
        print("[skip] PCR1", out)
        return

    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id",
        "locus",
        "length",
        "observed_multiplicity",
        "starting_copies",
        "pcr_efficiency",
        "pcr_copies",
    ]

    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)


def locus_read_budgets(sample):
    return {"IGH": int(READ_BUDGETS[sample])}


_FRAGMENT_MODEL = {}

def retained_fragment_model(L):
    """(starts, ends, λ, кусков на молекулу, отброшенная доля λ, cumsum λ) для длины L;
    λ — ожидаемое число кусков (start, end) на молекулу, прошедших size-selection."""
    L = int(L)
    if L not in _FRAGMENT_MODEL:
        s, e, E = expected_terminal_fragments(L)
        if abs(float((E * (e - s)).sum()) - L) > 1e-6 * L:
            raise AssertionError(f"fragmentation nucleotide-mass conservation failed for L={L}")
        lam = E * _size_selection_probability(e - s)
        keep = lam >= FRAGMENT_MODEL_MIN_EXPECTED
        _FRAGMENT_MODEL[L] = (s[keep], e[keep], lam[keep], float(E.sum()), float(lam[~keep].sum()),
                              np.cumsum(lam[keep]))
    return _FRAGMENT_MODEL[L]


def run_library_stage(sample, force=FORCE):
    """PCR1-пул → порция LIBRARY_INPUT_NG → фрагментация + size-selection → PCR2 → риды.

    1. Порция: A молекул (масса → число молекул), мультиномиально по численности после PCR1.
    2. Риды делятся между шаблонами пропорционально ожидаемому числу отобранных кусков
       (копии в порции × λ его длины). Средний выход PCR2 от длины куска не зависит и
       сокращается; отклонение суммы по тысячам кусков шаблона от ожидания пренебрежимо.
    3. Только для шаблонов с ридами разыгрывается пул кусков: Poisson(копии × λ) по видам
       (start, end), PCR2 каждого вида со своей эффективностью, риды шаблона —
       мультиномиально по копиям после PCR2. Так тысячи копий на молекулу не требуют
       перечисления миллиардов кусков.
    """
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fa = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_library_summary.tsv"
    model_out = FRAGMENTATION_DIR / f"{sample}_fragment_length_model.tsv"
    if all(p.exists() for p in (out_counts, out_fa, out_qc, summary_out, model_out)) and not force:
        print("[skip] library", sample)
        return

    rows = read_dict_rows(inp)
    for r in rows:
        r["length"] = int(r["length"])
        r["starting_copies"] = int(r["starting_copies"])
        r["pcr_copies"] = int(r["pcr_copies"])
    seqs = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    rng = np.random.default_rng(sample_seed(sample, 2000))

    lengths = np.array([r["length"] for r in rows])
    pcr1 = np.array([r["pcr_copies"] for r in rows], dtype=float)
    mean_len = float((pcr1 * lengths).sum() / pcr1.sum())
    n_input = int(LIBRARY_INPUT_NG * 1e-9 / (mean_len * DSDNA_G_PER_MOL_PER_BP) * AVOGADRO)
    aliquot = rng.multinomial(n_input, pcr1 / pcr1.sum())

    t0 = time.time()
    per_molecule = {int(L): float(retained_fragment_model(L)[2].sum()) for L in np.unique(lengths)}
    print(f"[{sample}] fragment model for {len(per_molecule)} lengths: {time.time() - t0:.0f} s", flush=True)
    weight = aliquot * np.array([per_molecule[int(L)] for L in lengths])

    loci = np.array([r["locus"] for r in rows])
    reads = np.zeros(len(rows), dtype=np.int64)
    budgets = locus_read_budgets(sample)
    for locus, n in budgets.items():
        if not n:
            continue
        idx = np.flatnonzero(loci == locus)
        if not len(idx) or weight[idx].sum() <= 0:
            raise RuntimeError(f"{sample}: no library molecules for locus {locus}")
        reads[idx] = rng.multinomial(n, weight[idx] / weight[idx].sum())

    mean, conc = LIBRARY_PCR_EFFICIENCY_MEAN, LIBRARY_PCR_EFFICIENCY_CONCENTRATION
    alpha, beta = mean * conc, (1 - mean) * conc
    fields = ["fragment_id", "template_id", "locus", "template_length", "fragment_start",
              "fragment_length", "fragment_input_copies", "library_pcr_efficiency",
              "library_pcr2_copies", "simulated_read_pairs"]
    tmp = {p: Path(str(p) + ".tmp") for p in (out_counts, out_fa, out_qc)}
    species_with_reads = fallback_templates = 0
    t0 = time.time()
    with open(tmp[out_counts], "w", newline="") as ch, open(tmp[out_fa], "w") as fh, \
         open(tmp[out_qc], "w", newline="") as qh:
        cw = csv.writer(ch, delimiter="\t")
        qw = csv.DictWriter(qh, fieldnames=fields, delimiter="\t")
        qw.writeheader()
        for done, i in enumerate(np.flatnonzero(reads), 1):
            r, m, n = rows[i], int(aliquot[i]), int(reads[i])
            s, e, lam, _, _, cdf = retained_fragment_model(r["length"])
            # Пул кусков шаблона: Poisson(m·λ) по видам (start, end). Если ожидаемых кусков
            # меньше, чем видов, то же распределение быстрее: K ~ Poisson(m·Σλ), виды — K выборок по λ.
            if m * cdf[-1] < len(lam):
                picks = np.searchsorted(cdf, rng.random(int(rng.poisson(m * cdf[-1]))) * cdf[-1], side="right")
                nz, k = np.unique(picks, return_counts=True)
            else:
                k = rng.poisson(m * lam)
                nz = np.flatnonzero(k)
                k = k[nz]
            if not len(k):
                # ponytail: у шаблона с 1–2 копиями в порции ни один кусок не прошёл отбор, а риды
                # по ожиданию есть — берём один кусок по λ; при реалистичной порции почти не бывает.
                nz = np.searchsorted(cdf, rng.random(1) * cdf[-1], side="right")
                k = np.ones(1, dtype=np.int64)
                fallback_templates += 1
            s, e = s[nz], e[nz]
            eff = np.ones(len(k)) if mean == 1 else rng.beta(alpha, beta, size=len(k))
            cp = branching_pcr_vectorized(k, eff, LIBRARY_PCR_CYCLES, LIBRARY_PCR_MAX_COPIES, rng)
            pairs = rng.multinomial(n, cp / cp.sum())
            seq = seqs[r["template_id"]]
            for j in np.flatnonzero(pairs):
                start, end = int(s[j]), int(e[j])
                fid = f"{r['template_id']}_{FRAGMENT_ID_TAG}_{start}_{end}"
                # InSilicoSeq readcount_file ожидает число ридов, поэтому пары умножаются на 2.
                cw.writerow([fid, int(pairs[j]) * 2])
                fh.write(f">{fid}\n{seq[start:end]}\n")
                qw.writerow({"fragment_id": fid, "template_id": r["template_id"], "locus": r["locus"],
                             "template_length": r["length"], "fragment_start": start,
                             "fragment_length": end - start, "fragment_input_copies": int(k[j]),
                             "library_pcr_efficiency": float(eff[j]), "library_pcr2_copies": int(cp[j]),
                             "simulated_read_pairs": int(pairs[j])})
                species_with_reads += 1
            if done % 200000 == 0:
                print(f"  [{sample}] templates with reads done: {done:,} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    for p, t in tmp.items():
        t.replace(p)

    total_pairs = int(reads.sum())
    assert total_pairs == sum(budgets.values())
    starting_total = sum(r["starting_copies"] for r in rows)
    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_templates": len(rows),
        "pcr1_molecules_total": int(pcr1.sum()),
        "starting_copies_total": starting_total,
        "library_input_ng": LIBRARY_INPUT_NG,
        "pcr1_weighted_mean_template_length": round(mean_len, 1),
        "library_input_molecules": n_input,
        "library_input_copies_per_starting_molecule": round(n_input / starting_total, 1),
        "library_input_templates": int((aliquot > 0).sum()),
        "expected_retained_fragments_per_input_molecule": round(float(weight.sum() / aliquot.sum()), 4),
        "templates_with_reads": int((reads > 0).sum()),
        "read_pairs": total_pairs,
        "fragment_species_with_reads": species_with_reads,
        "read_pairs_per_fragment_species": round(total_pairs / species_with_reads, 3),
        "templates_single_fragment_fallback": fallback_templates,
        "size_selection_target_nt": SIZE_SELECTION_TARGET,
        "size_selection_sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_fragment_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    }
    write_dict_rows(summary_out, [summary], list(summary))
    model_rows = []
    for L, cnt in zip(*np.unique(lengths, return_counts=True)):
        s, e, lam, pieces, pruned, _ = retained_fragment_model(L)
        model_rows.append({"template_length": int(L), "templates": int(cnt),
                           "expected_pieces_per_molecule": round(pieces, 4),
                           "expected_retained_per_molecule": round(float(lam.sum()), 6),
                           "mean_retained_length": round(float((lam * (e - s)).sum() / lam.sum()), 1),
                           "retained_species": len(lam), "pruned_expected": pruned})
    write_dict_rows(model_out, model_rows, list(model_rows[0]))
    print(f"[{sample}] library: {summary}")

for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_library_stage(sample)


## 12. Сводка библиотеки


In [ ]:
library_summary = [read_dict_rows(FRAGMENTATION_DIR / f"{sample}_library_summary.tsv")[0] for sample in SAMPLES]
out = QC_DIR / "library_summary.tsv"
write_dict_rows(out, library_summary, list(library_summary[0]))
print(*library_summary, sep="\n")


## 12a. Восстановимость эталонных шаблонов (reconstructability)

Для каждого эталонного шаблона по фрагментам, получившим риды, определяется, что сборщик **в принципе** может восстановить в интервале V…J:

| класс | смысл |
|---|---|
| `no_reads` | V…J шаблона не попал в FASTQ |
| `partial_read_coverage` | часть V…J не прочитана ни одним ридом |
| `full_coverage_unlinked` | всё прочитано, но есть точка, которую не пересекает ни один фрагмент: куски физически не связаны |
| `full_coverage_mate_linked_only` | каждая точка пересечена фрагментом, но хотя бы одна — только промежутком между R1 и R2 (mate-pair extension) |
| `full_coverage_read_overlap` | каждая точка внутри рида с `RECONSTRUCT_MIN_FLANK_NT` с обеих сторон — сборка обычным перекрытием |

Эти классы — знаменатели для benchmark сборщиков.


In [ ]:
# Класс восстановимости эталонного интервала V…J каждого шаблона по тому, что реально попало в FASTQ.
# ISS amplicon: R1 = [start, start+R), R2 = [end-R, end); вся вставка — физическая связь.
# Связь (bond) b между позициями b-1 и b:
#   read_overlap — b лежит внутри одного рида с >= K нт с каждой стороны
#                  (собирается обычным перекрытием);
#   mate_only    — b внутри фрагмента, но в несеквенированном промежутке между
#                  R1 и R2 (соединить может только mate-pair extension TRUST4);
#   unlinked     — ни один фрагмент не пересекает b: куски шаблона физически
#                  не связаны, соединить их нельзя без внешней информации.
RECONSTRUCT_MIN_FLANK_NT = 31  # ponytail: грубый аналог k-mer/min-overlap сборщика, калибруемый

def _bond_mask(L, intervals):
    d = np.zeros(L + 2, dtype=np.int32)
    for a, b in intervals:
        a, b = max(a, 1), min(b, L - 1)
        if a <= b:
            d[a] += 1
            d[b + 1] -= 1
    return np.cumsum(d)[: L + 1] > 0

def classify_template(L, frags, R=TARGET_READ_LENGTH, K=RECONSTRUCT_MIN_FLANK_NT, region=None):
    """frags: [(start, end)] фрагментов с ридами в координатах шаблона.

    region=(vj_start, vj_end): оценивается только эталонный интервал V…J
    (праймеры и leader/C-фланги не требуют сборки). Возвращает (class, read_covered_fraction).
    """
    reads = [iv for s, e in frags for iv in ((s, min(s + R, e)), (max(e - R, s), e))]
    if region is not None:
        a0, b0 = region
        clip = lambda ivs: [(max(a, a0) - a0, min(b, b0) - a0) for a, b in ivs if min(b, b0) > max(a, a0)]
        L, frags, reads = b0 - a0, clip(frags), clip(reads)
    if not frags:
        return "no_reads", 0.0
    cov = np.zeros(L + 1, dtype=np.int32)
    for a, b in reads:
        cov[a] += 1
        cov[b] -= 1
    covered = float((np.cumsum(cov)[:L] > 0).mean())
    if covered < 1.0:
        return "partial_read_coverage", covered
    by_frag = _bond_mask(L, [(s + K, e - K) for s, e in frags])
    by_read = _bond_mask(L, [(a + K, b - K) for a, b in reads])
    bonds = slice(K, L - K + 1)  # связи ближе K к краям шаблона не требуют соединения
    if not by_frag[bonds].all():
        return "full_coverage_unlinked", covered
    if not by_read[bonds].all():
        return "full_coverage_mate_linked_only", covered
    return "full_coverage_read_overlap", covered

# self-check: 400 нт, PE150
assert classify_template(400, [])[0] == "no_reads"
assert classify_template(400, [(0, 200), (200, 400)])[0] == "full_coverage_unlinked"
assert classify_template(300, [(0, 200), (100, 300)])[0] == "full_coverage_read_overlap"
assert classify_template(400, [(0, 400)])[0] == "partial_read_coverage"          # середина между R1 и R2 не прочитана
assert classify_template(300, [(0, 300)])[0] == "full_coverage_mate_linked_only"  # R1 и R2 стыкуются без перекрытия
assert classify_template(420, [(0, 420), (100, 320)])[0] == "full_coverage_mate_linked_only"
# фланги вне V…J не учитываются; фрагмент только в праймере = нет ридов в V…J
assert classify_template(500, [(0, 300), (200, 500)], region=(50, 450))[0] == "full_coverage_mate_linked_only"
assert classify_template(500, [(0, 40)], region=(50, 450))[0] == "no_reads"

def write_reconstructability_qc(sample):
    frags = defaultdict(list)
    for r in read_dict_rows(ALLOCATION_DIR / f"{sample}_allocation.tsv"):
        if int(r["simulated_read_pairs"]) > 0:
            s = int(r["fragment_start"])
            frags[r["template_id"]].append((s, s + int(r["fragment_length"])))
    rows, summary = [], defaultdict(lambda: defaultdict(int))
    for t in load_dedup_templates(sample):
        cls, cov = classify_template(t["length"], frags.get(t["template_id"], []),
                                     region=(t["vj_start"], t["vj_end"]))
        rows.append({"template_id": t["template_id"], "locus": t["locus"], "length": t["length"],
                     "observed_multiplicity": t["observed_multiplicity"],
                     "fragments_with_reads": len(frags.get(t["template_id"], [])),
                     "read_covered_fraction": round(cov, 4), "reconstructability": cls, "verified": t["verified"]})
        if t["verified"]:  # знаменатели оценки сборщиков — только проверенные шаблоны
            summary[t["locus"]][cls] += 1
    write_dict_rows(QC_DIR / f"{sample}_template_reconstructability.tsv", rows, list(rows[0]))
    classes = ["no_reads", "partial_read_coverage", "full_coverage_unlinked",
               "full_coverage_mate_linked_only", "full_coverage_read_overlap"]
    out = []
    for locus in sorted(summary) + ["ALL"]:
        c = {k: (sum(summary[l][k] for l in summary) if locus == "ALL" else summary[locus][k]) for k in classes}
        n = sum(c.values())
        out.append({"sample": sample, "locus": locus, "templates": n, **c,
                    **{f"{k}_frac": round(c[k] / n, 4) for k in classes}})
    write_dict_rows(QC_DIR / f"{sample}_reconstructability_summary.tsv", out, list(out[0]))
    print(*out, sep="\n")

for sample in SAMPLES:
    write_reconstructability_qc(sample)


## 13. Генерация ридов PE150 с профилем NovaSeq


In [ ]:
def crop_fastq(in_path,out_path,length):
    tmp=Path(str(out_path)+".tmp")
    with open_text(in_path,"rt") as src,gzip.open(tmp,"wt") as dst:
        n=0
        while True:
            h=src.readline()
            if not h: break
            s=src.readline().rstrip("\n\r"); p=src.readline(); q=src.readline().rstrip("\n\r")
            if len(s)<length or len(q)<length: raise ValueError(f"short read in {in_path}: {len(s)}")
            dst.write(h); dst.write(s[:length]+"\n"); dst.write(p); dst.write(q[:length]+"\n"); n+=1
    tmp.replace(out_path); return n

def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; native_prefix=FASTQ_NATIVE_DIR/sample; final_prefix=FASTQ_DIR/sample; ext=".fastq.gz"
    nr1=Path(str(native_prefix)+f"_R1{ext}"); nr2=Path(str(native_prefix)+f"_R2{ext}"); r1=Path(str(final_prefix)+f"_R1{ext}"); r2=Path(str(final_prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",ISS_PROFILE_NAME,"--cpus",str(NPROC),"--output",str(native_prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_novaseq.log")
    n1=crop_fastq(nr1,r1,TARGET_READ_LENGTH); n2=crop_fastq(nr2,r2,TARGET_READ_LENGTH); assert n1==n2==READ_BUDGETS[sample]
    print("NovaSeq native PE151 -> exact PE150:",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":READ_BUDGETS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==READ_BUDGETS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и ограничения

- Симуляция относится только к ветке предобработки `ERP003950`.
- Источник набора истины — собственный `post_annotation_filtered/airr_pass` этой ветки.
- В датасете нет UMI: ошибки исходных ридов схлопываются (1 замена, сосед ≥3× чаще), исходные копии = риды после схлопывания. Это включает PCR-искажения исходной библиотеки; шаблоны с 1 ридом в оценку не входят.
- Точная наблюдаемая V…J-последовательность сохраняет настоящую SHM; germline-последовательность не подставляется.
- Шаблон — физический ампликон PCR1 (склеенный рид с V-праймером + revcomp C-праймера); эталон для оценки сборки — интервал `vj_start…vj_end`. Вырожденные позиции C-праймера разрешены случайно, но воспроизводимо.
- `LIBRARY_INPUT_NG`, параметры схлопывания и `READ_BUDGET_MULTIPLIER` входят в имя ветки.
- PCR1/PCR2 представлены стохастическими ветвящимися моделями; ошибки полимеразы, химеры и истощение реагентов явно не моделируются.
- Для ошибок секвенирования используется встроенная модель InSilicoSeq NovaSeq. Нативные риды PE151 детерминированно обрезаются до PE150.
- Бюджет секвенирования каждого sample по умолчанию берётся из исходных R1/R2 в `raw/ERP003950`, поскольку ветка fastp представляет альтернативную предобработку того же физического датасета.
- Фрагментация использует один равновероятный random cut на выбранную физическую молекулу; оба дочерних фрагмента создаются до size-selection.
- В фрагментацию идёт `LIBRARY_INPUT_NG` нг продукта PCR1 (тысячи копий на исходную молекулу); модель кусков по длине проверяется на сохранение массы нуклеотидов. Выход PCR2 шаблона берётся по ожиданию (сумма по тысячам кусков), разброс PCR2 между кусками моделируется.
